# Assignment Data Preparation

This notebook extracts the assignment data used for the analysis and prepares it for the later combination with the caregiver and client datasets. The data is retrieved from the source system, flattened into a tabular structure and cleaned before it is exported for further processing.

In [ ]:
import pandas as pd
import json
import numpy as np
import requests
from tqdm.auto import tqdm
from config import URL, TOKEN
from datetime import datetime
import urllib3
from dateutil.relativedelta import relativedelta
import time
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## Assignment Data Extraction

The assignment data is requested from the source system for the period starting in 2017. To reduce the size of the individual requests, the data is fetched month by month and in multiple pages where necessary. Only the relevant assignment types are requested. Failed requests are retried before the respective request is skipped.

In [ ]:
headers = {
    "Authorization": f"Bearer {TOKEN}"
}

start_date = datetime(2017, 1, 1)
today = datetime(2026, 1, 1)

all_data = {"entries": []}

total_months = (
    (today.year - start_date.year) * 12
    + today.month - start_date.month
)

max_retries = 3

current = start_date

progress = tqdm(
    total=total_months,
    desc="Fetching months",
    unit="month"
)

while current < today:

    next_month = current + relativedelta(months=1)

    month_end = min(next_month, today)

    date_from = current.strftime("%Y%m%d000000")
    date_to = month_end.strftime("%Y%m%d%H%M%S")

    itemCount = np.inf
    offset = 0
    steps = 2000

    while offset < itemCount:

        url = (
            f"{URL}assignment/query?"
            f"filter="
            f"assignmentType:in:24h;spring;first,"
            f"date:gte:{date_from},"
            f"date:lt:{date_to}"
            f"&sort=startDate.timeUtc"
            f"&limit={steps}"
            f"&skip={offset}"
        )

        request_successful = False

        for attempt in range(1, max_retries + 1):

            try:
                response = requests.get(
                    url,
                    headers=headers,
                    verify=False,
                    timeout=60
                )

                if response.status_code == 200:
                    request_successful = True
                    break

                tqdm.write(
                    f"Request failed ({response.status_code}) "
                    f"for {date_from} -> {date_to}, offset={offset} "
                    f"[attempt {attempt}/{max_retries}]"
                )

            except requests.RequestException as e:
                tqdm.write(
                    f"Request error for {date_from} -> {date_to}, "
                    f"offset={offset}: {e} "
                    f"[attempt {attempt}/{max_retries}]"
                )

            # Wait before next retry
            if attempt < max_retries:
                time.sleep(5)

        # All retries failed
        if not request_successful:
            tqdm.write(
                f"Giving up after {max_retries} attempts: "
                f"{date_from} -> {date_to}, offset={offset}"
            )
            break

        responseData = response.json()
        entries = responseData.get("entries", [])

        itemCount = responseData["pagingInfo"]["itemCount"]

        all_data["entries"].extend(entries)

        progress.set_postfix(
            collected=f"{len(all_data['entries']):,}",
            current_month=current.strftime("%Y-%m")
        )

        if len(entries) == 0:
            break

        offset += len(entries)

    progress.update(1)
    current = next_month

progress.close()

print(f"Total assignments collected: {len(all_data['entries']):,}")

## Feature Extraction

The returned assignment data contains nested information which cannot directly be used as a tabular dataset. Features which are not required for the analysis, including identifying and administrative information, are excluded. The remaining values are extracted from the nested structure depending on their respective data type and answer scheme.

In [ ]:
exceptions = [
	"metadata",
	"locality",
	"postalcode",
	"contactoptions",
	"personFullName",
	"personFullNameNoTitle",
	"tenantid",
	"persontype",
	"adresse",
	"mailAddressing",
	"socialInsuranceNumber",
	"comments",
	"givenName",
	"familyName",
	"primaryEmailAddress",
	'addresses',
	'primaryPhoneNumber',
	'address',
	'residentialAddress',
	'address',
	'geoLocation',
	'billingAddress',
	'serviceAddress',
	'legalAddress',
	'businessAddress',
	'emailAddresses',
	'phoneNumbers',
	'personFullNameNoTitle',
	'personFullName',
	'mailAddressing',
	'postalAddress',
	'contractualAddressing',
	'caatsDateOfBirth',
	'gender',
	'locations',
	'bankDetails',
	'personStatus',
	"chapterId",
	"language",
	"ordinal",
	"sectionId",
	"academicTitlePrefix",
	"personNr",
	"content",
	"consecutiveNumber",
	'Klient:innen_Empfohlen_Ja',
	'Klient:innen_Erstrkontakt_erfassen_Ja',
	'Klient:innen_Bew_Ein_Ja',
	'abrech-akonto',
	'abrech-buerge-hinterlegt',
	'pers-visite-keine',
	'medizinische-delegation-hochgeladen',
	'medizinische-delegation-nicht-notwendig',
	'Klient:innen_Medizinische_Delegation_Ja',
	'pflegerische-delegation-hochgeladen',
	'pflegerische-delegation-nicht-notwendig',
	'Klient:innen_Pflegerische_Delegation_JA',
	"admin-beruf",
	'pflegevisite-letzte-date',
	'pflegevisite-letzte-dgkp',
	'pflegerisch-person',
	'erstkontakt-bemerkung',
	"admin-kooperationspartner",
	'birthName',
'confessionId',
'confession',
'nationalityId',
'placeOfBirth',
'paymentBlockReason',
'chamberOfCommerceMembershipNr',
'avatarFileId',
'empfohlen-von-category',
'visibilityConditionId',
'deutschkennt-bew-von',
'Kinder_Nein',
'covid-1',
'covid-2',
'covid-3',
'keine-kurse',
'agentur-zuletzt',
'orgRef',
'businesscase',
'comment',
'commentArrival',
'commentDeparture',
'region',
'contactInfo',
'busnessCaseStatus',
'transportOrgArrival',
'transportOrgDeparture',
'assigneeContactInfo',
'assigneeNationality',
'assigneOrgRef',
'assignmentType',
'businessCaseOrgRef',
'businessCaseType',
'assigneeOrgRef',
'hasOrder',
'businessCaseStatus',
"businessCaseStartDate",
"businessCaseEndDate",
"clientAddress",
"region",
"arrivalDate",
"departureDate",
"timeUtc",
"objectKey",
"categoryShortName",
"hasDisplayText",
"displayText",
"objectId",
"objectType",
"cycleLength"
]

exceptions = [x.lower() for x in exceptions]

In [ ]:
def extractStatement(statement):
    field = {}
    if statement["statementId"].lower() in exceptions:
        return field

    match statement["answerScheme"]:
        case 1:
            field[statement["statementId"]] = statement["answerValue"]["displayText"]
        case 2:
            #do nothing
            field = {}
        case 3:
            field[statement["statementId"]] = statement["answerYesno"] == 2
        case 4:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False

        case 5:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False
        case 8:
            if "answerDateTime" in statement.keys():
                if "userLocalTime" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["userLocalTime"]
                elif "timeUtc" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["timeUtc"]
        case _:
            print(f'Found statement: {statement["answerScheme"]} - {statement["statementId"]} -  {statement}')


    return field
        

def returnEntry(entry):
    keys = entry.keys()
    finalFields = {}
    if "statementId" in keys:
        finalFields = finalFields | extractStatement(entry)
    else:
        for key in keys:
            
            if key.lower() in exceptions:
                continue
            
            typing = type(entry[key]).__name__
            match typing:
                case "str":
                    finalFields[key] = entry[key]
                    
                case "int":
                    finalFields[key] = entry[key]

                case "bool":
                    finalFields[key] = entry[key]

                case "float":
                    finalFields[key] = entry[key]
                    
                case "dict":
                    if key == "assignee":
                        finalFields[key] = entry[key]["objectId"]
                        continue
                    if "displayText" in entry[key].keys():
                        finalFields[key] = entry[key]["displayText"]
                        continue
                    if "userLocalTime" in entry[key].keys():
                        finalFields[key] = entry[key]["userLocalTime"]
                        continue
                    if "timeUtc" in entry[key].keys():
                        finalFields[key] = entry[key]["timeUtc"]
                        continue
                    else:
                        finalFields = finalFields | returnEntry(entry[key])
                    
                case "list":
                    if key == "clients":
                        finalFields[key] = entry[key][0]["objectId"]
                    else:
                        for item in entry[key]:
                            if type(item).__name__ == "dict":
                                finalFields = finalFields | returnEntry(item)
                        

                case _:
                    print(typing)


    
    return finalFields

## Creation and Initial Cleaning of the Assignment Dataset

The extracted entries are combined into a DataFrame and duplicate rows are removed. Boolean features are converted to a consistent format and missing Boolean values are set to `False`. Empty strings are treated as missing values. At this stage, the resulting dataset is also exported as a raw assignment dataset before the remaining feature selection steps are applied.

In [ ]:
totalData = []

if all_data["entries"] is not None:
    for entry in all_data["entries"]:
        test = returnEntry(entry)
        totalData.append(test)
        
df = pd.DataFrame(totalData)
df.drop_duplicates(inplace=True)

## Export

After the cleaning and feature selection steps, the prepared assignment dataset is exported and used in the following data preparation and aggregation steps.

In [ ]:
for col in df.columns:
    non_null = df[col].dropna()

    if len(non_null) > 0 and non_null.isin([True, False]).all():
        df[col] = df[col].fillna(False).astype(bool)

bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns

df[bool_cols] = df[bool_cols].fillna(False)


df = df.replace(r'^\s*$', np.nan, regex=True)

df.to_csv("assignmentsRaw.csv", sep=";", index=False)